In [ ]:
from THBSplines.src.hierarchical_space import HierarchicalSpace
import numpy as np
import scipy.sparse as sp
import dolfinx
from mpi4py import MPI
import basix.ufl
import pyvista

import cffi
import numba
import numba.core.typing.cffi_utils as cffi_support
from dolfinx.jit import ffcx_jit
from dolfinx import default_real_type, default_scalar_type, geometry
rtype = default_real_type
dtype = default_scalar_type
import ufl
from ffcx.codegeneration.utils import empty_void_pointer
from ffcx.codegeneration.utils import numba_ufcx_kernel_signature as ufcx_signature

from typing import Dict
import numpy.typing as npt

In [ ]:
def refine(knots: npt.ArrayLike, p: int, n_times: int=1)->npt.NDArray:
    """Given `knots`, returns its dyadic refinement with multiplicity `p+1`
    at the extremities."""
    knots: npt.NDArray[np.float_] = np.asarray(knots)
    mult_left: int = np.searchsorted(knots, knots[0], side='right')
    mult_right: int = len(knots) - np.searchsorted(knots, knots[-1], side='left')
    pad_left: int = max(0, p + 1 - mult_left)
    pad_right: int = max(0, p + 1 - mult_right)
    if pad_left > 0 or pad_right > 0:
        knots = np.concatenate((
            np.full(pad_left, knots[0], dtype=knots.dtype),
            knots,
            np.full(pad_right, knots[-1], dtype=knots.dtype)
        ))
    if n_times == 0:
        return knots
    
    # Find indices where the knot value changes
    jump_idx: npt.NDArray[np.int_] = np.where(knots[1:] > knots[:-1])[0]
    left_vals = knots[jump_idx]
    right_vals =knots[jump_idx + 1]
    num_new_points = (1<<n_times)-1
    fractions = np.linspace(0.,1.,num_new_points+2)[1:-1]
    new_points = left_vals[:, None] + (right_vals - left_vals)[:, None] * fractions[None, :]
    new_points = new_points.ravel()
    insert_positions = np.repeat(jump_idx + 1, num_new_points)
    
    return np.insert(knots, insert_positions, new_points)
        

In [ ]:
p0, p1 = 2,2
knots1 = np.array([-4., 0., 4.], dtype=np.float64)
knots1 = refine(knots1, p=p0, n_times=1)
knots2 = knots1# np.array([0, 0,0, 1, 1., 1], dtype=np.float64)
hs = HierarchicalSpace(knots=[knots1, knots1], degrees=[p0])
hs.refine(marked_cells=[5,6, 9, 10], level=0)
#hs.refine(marked_cells=[9], level=1)
hs.mesh.plot_cells()

In [ ]:
all_cells = np.array([[]], dtype=np.float64).reshape(-1, 2) # will have coarser cells on top and finer on bottom
thb_operators: dict[tuple[int, int], sp.csr_array] = {}
N_max = 0 # maximum amount of dofs in a cell
for l in range(hs.nlevels):
    active_cells_l = hs.mesh.aelem_level[l]
    my_cells_l = hs.mesh.meshes[l].cells[active_cells_l]
    for cell in active_cells_l:
        thb_operators[l, cell] = hs.local_multi_level_extraction_operator(element_idx=cell, element_level=l, l=l)
        #print(f"l={l}, cell={cell}, size = {thb_operators[l, cell].shape}")
        N_max = max(N_max, thb_operators[l, cell].shape[0])
    pass

    x_coords = my_cells_l[:, 0, :]
    y_coords = my_cells_l[:, 1, :] 

    points = np.zeros((4 * len(my_cells_l), 2), dtype=np.float64)
    points[::4] = np.column_stack((x_coords[:, 0], y_coords[:, 0]))  # Bottom-left
    points[1::4] = np.column_stack((x_coords[:, 1], y_coords[:, 0]))  # Bottom-right
    points[2::4] = np.column_stack((x_coords[:, 0], y_coords[:, 1]))  # Top-left
    points[3::4] = np.column_stack((x_coords[:, 1], y_coords[:, 1]))  # Top-right

    all_cells = np.vstack((all_cells, points))
pass

In [ ]:
all_cells = np.array(all_cells).reshape(-1, 2)

cells = np.arange(len(all_cells), dtype=np.int32).reshape(-1, 4)
coordinate_element = basix.ufl.element("Q", "quadrilateral", 1, shape=(2,))
disconnected_mesh = dolfinx.mesh.create_mesh(MPI.COMM_WORLD, cells, coordinate_element, all_cells)

In [ ]:
topology, cell_types, geometry = dolfinx.plot.vtk_mesh(disconnected_mesh)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

plotter = pyvista.Plotter()
plotter.add_mesh(grid.shrink(.9), show_edges=False, color="lightblue")
plotter.view_xy()
plotter.show(jupyter_backend="static")
print(f"Number of points in PyVista grid: {grid.n_points}")

In [ ]:
legendre_elt = basix.ufl.element(
    "DG",
    "quadrilateral",
    degree=p0,
    lagrange_variant=basix.LagrangeVariant.legendre
)
V = dolfinx.fem.functionspace(disconnected_mesh, legendre_elt)
print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")

u,v = ufl.TrialFunction(V), ufl.TestFunction(V) 
f = dolfinx.fem.Function(V)
sigma = 0.9
# Ricker wavelet
f.interpolate(lambda x: 1./(np.pi*sigma**4)*(1.-0.5*((x[0]**2+x[1]**2)/sigma**2)) * np.exp(-(x[0]**2+x[1]**2)/(2.*sigma**2)))
a0 = ufl.inner(u, v) * ufl.dx
f0 = ufl.inner(f, v)*ufl.dx

msh = disconnected_mesh
ufcxa0, _, _ = ffcx_jit(msh.comm, a0, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernela0 = getattr(ufcxa0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

ufcxf0, _, _ = ffcx_jit(msh.comm, f0, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernelf0 = getattr(ufcxf0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

ffi = cffi.FFI()

In [ ]:
dofmap, dummy_dof_index = hs.build_global_dof_map()
dummy_dof_index += 1 
# does not exist, and we will map unwanted contributions to this function

In [ ]:
num_control_points_with_dummy = dummy_dof_index+1

num_cells = disconnected_mesh.topology.index_map(disconnected_mesh.topology.dim).size_global

# which BSpline functions are active on each cell, padded with the dummy index.
padded_cells_to_dofs = np.full((num_cells, N_max), dummy_dof_index, dtype=np.int32)

# Find the maximum local index for each level to size arrays
max_local_indices = [0] * hs.nlevels
for l, local_idx in dofmap.keys():
    if local_idx > max_local_indices[l]:
        max_local_indices[l] = local_idx
    pass
pass

# convert the dictionary into a list of numpy arrays for faster lookup
dofmap_arrays = [np.full(size + 1, dummy_dof_index, dtype=np.int32) for size in max_local_indices]
for (l, local_idx), global_idx in dofmap.items():
    dofmap_arrays[l][local_idx] = global_idx


cell_index = 0
for l in range(hs.nlevels):
    active_cells_l = hs.mesh.aelem_level[l]
    for cell in active_cells_l:
        
        # Get the local functions on the cell
        active_funcs_dict: dict[int, npt.NDArray[np.int_]] = hs.get_all_active_functions_on_cell(l, cell)
        
            
        #  Vectorized conversion from local to global indices
        mapped_arrays = [
            dofmap_arrays[ll][local_funcs] for ll, local_funcs in active_funcs_dict.items() if len(local_funcs) > 0
        ]
        if l<0:
            print(f"level={l}, cell={cell}")
            print(f"active funcs = \n {active_funcs_dict}")
            print(f"mapped arrays = \n {mapped_arrays} \n\n")
        
        #
        if mapped_arrays: # Check if there are actually active functions
            global_dofs = np.concatenate(mapped_arrays)
            n_dofs = len(global_dofs)
            padded_cells_to_dofs[cell_index, :n_dofs] = global_dofs
            
        cell_index += 1


In [ ]:
M = hs._bezier_to_legendre(degree = p0)
S_indices = np.arange(p0+1, dtype=np.float64)
# scaling for unnormalised Legendre basis polynomials
S_inv = 1./np.sqrt(2.*S_indices+1.)#*np.identity(p0+1, dtype=np.float64) # Scaling factor, since fenicsx uses orthonormal legendre polynomials
T = np.asfortranarray(np.kron(M, M).T * np.kron(S_inv, S_inv).reshape(-1,1), dtype=dtype)
local_dofs_size = T.shape[1]

operator_shape = (N_max, local_dofs_size)
# Create a custom space that holds the content of each matrix for the relevant cell.
# degree 0 because the value is constant over each cell
C_space = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0, operator_shape))
C_func = dolfinx.fem.Function(C_space, dtype=dtype)

num_cells_local = disconnected_mesh.topology.index_map(disconnected_mesh.topology.dim).size_local
indices = np.arange(num_cells_local, dtype=np.int32)
# To make sure that each matrix is assigned to the correct cell
midpoints: npt.NDArray[np.float_] = dolfinx.mesh.compute_midpoints(disconnected_mesh, disconnected_mesh.topology.dim, indices)


In [ ]:
multi_level_bezier = []
for ll in range(hs.nlevels):
    active_cells_l = hs.mesh.aelem_level[ll]
    for cell in active_cells_l:
        # print(f"l={ll}, cell = {cell}")
        mat = thb_operators[ll, cell].dot(hs.bezier_operators[ll][cell])
        real_k, n_cols = mat.shape
        if real_k<N_max:
            padding_size = N_max - real_k
            # add rows of zeros below the result
            new_indptr = np.concatenate([
                mat.indptr, 
                np.full(padding_size, mat.indptr[-1], dtype=mat.indptr.dtype)
            ])
            mat_padded = sp.csr_array(
                (mat.data, mat.indices, new_indptr), 
                shape=(N_max, n_cols)
            )
            multi_level_bezier.append(mat_padded)
        else:
            multi_level_bezier.append(mat)
    pass
pass

In [ ]:
c_values = C_func.x.array.reshape((-1, N_max, T.shape[1]))
for local_idx, midpoint in enumerate(midpoints):
    #print(f"midpoint = {midpoint}")
    level, idx = hs.mesh.find_active_cell(midpoint[:hs.dim])

    mat: sp.csr_array = thb_operators[level, idx].dot(hs.bezier_operators[level][idx])
    real_k, n_cols = mat.shape

    if real_k<N_max:
        padding_size = N_max - real_k
        # add rows of zeros below the result
        new_indptr: npt.NDArray[np.int_] = np.concatenate([
            mat.indptr, 
            np.full(padding_size, mat.indptr[-1], dtype=mat.indptr.dtype)
        ])
        mat_padded = sp.csr_array(
            (mat.data, mat.indices, new_indptr), 
            shape=(N_max, n_cols)
        )
        Ci = mat_padded
    else:
        Ci = mat
    
    # is a view of C_func.x.array, therefore we modify the content of C_func.x.array
    # No new array is created, the matrix->cell mapping is done here.
    c_values[local_idx, :, :] = Ci@T
C_func.x.scatter_forward()

In [ ]:
my_index_map = dolfinx.common.IndexMap(comm=disconnected_mesh.comm, 
                                       local_size=num_control_points_with_dummy)

dummy_element = basix.ufl.element(
    family="DG", 
    cell="quadrilateral", 
    degree=0, 
    shape=(N_max,)
)
dummy_space = dolfinx.fem.functionspace(mesh=disconnected_mesh, 
                                        element=dummy_element)
element_layout = dummy_space.dofmap.dof_layout
cpp_element = dummy_space.element._cpp_object

dof_indices = padded_cells_to_dofs.ravel().astype(np.int32)
offsets = (np.arange(num_cells + 1, dtype=np.int32) * N_max).astype(np.int32)

adj = dolfinx.cpp.graph.AdjacencyList_int32(data=dof_indices, 
                                            offsets=offsets)

dofmap = dolfinx.cpp.fem.DofMap(
    element_dof_layout=element_layout, 
    index_map=my_index_map,  
    index_map_bs=1, 
    dofmap=adj, 
    bs=1
)

V_spline_cpp = dolfinx.cpp.fem.FunctionSpace_float64(
    mesh=disconnected_mesh._cpp_object, 
    element=cpp_element, 
    dofmap=dofmap
)

V_spline = dolfinx.fem.FunctionSpace(
    mesh=disconnected_mesh, 
    element=dummy_element, 
    cppV=V_spline_cpp
)

In [ ]:
PADDED_DOFS = N_max
LOCAL_DOFS = local_dofs_size

@numba.cfunc(ufcx_signature(dtype, rtype), nopython=True)  # type: ignore
def tabulate_A(A_, w_, c_, coords_, entity_local_index, permutation=ffi.NULL, custom_data=None):

    # Prepare target condensed local element tensor
    # arguments: ptr, shape, dtype
    # returns a view over the original array A_
    # This has to be larger since we are working with padded arrays. Irrelevant dofs are mapped to a dummy location
    A = numba.carray(A_, (PADDED_DOFS, PADDED_DOFS), dtype=dtype)

    # Get the operator (TRUNC @ C_{B->BS} @ (C_{L->B}.T) @ S^{-1}) for this cell
    # TRUNC has shape (PADDED_DOFS, LOCAL_DOFS) and all other matrices have shape (LOCAL_DOFS, LOCAL_DOFS)
    G = numba.carray(w_, (PADDED_DOFS, LOCAL_DOFS), dtype=dtype)
    
    # Tabulate all sub blocks locally
    # This matrix is formed via the Legendre elements on a quadrilateral of degree p0,
    # therefore this has the shape (LOCAL_DOFS, LOCAL_DOFS)
    A0 = np.zeros((LOCAL_DOFS, LOCAL_DOFS), dtype=dtype)
    kernela0(
        ffi.from_buffer(A0),
        w_,# weights. Ignored for the L2 approximation because 
        # a0 does not depend on f and no value is looked up at the quadrature points
        # This is not very robust, one must be careful when calling this function
        c_,
        coords_,
        entity_local_index,
        permutation,
        empty_void_pointer(),
    )
    
    A[:, :] = G@A0@(G.T)

@numba.cfunc(ufcx_signature(dtype, rtype), nopython=True)
def tabulate_b(b_, w_, c_, coords_, entity_local_index, permutation=ffi.NULL, custom_data=None):
    
    # Prepare target condensed local element tensor
    # arguments: ptr, shape, dtype
    # evaluation of f using padded THB-Splines
    b = numba.carray(b_, (PADDED_DOFS,), dtype=dtype)

    total_coeffs = LOCAL_DOFS*(PADDED_DOFS+1)
    # Evaluation of f at relevant points + values of change of basis matrix G
    w_flat = numba.carray(w_, (total_coeffs), dtype=dtype)

    # The following line breaks in case we do not pass [f._cpp_object, C_func._cpp_object] in this
    # exact order or if f does not have the correct amount of dofs.
    G = w_flat[LOCAL_DOFS:].reshape((PADDED_DOFS, LOCAL_DOFS))

    b0 = np.zeros((LOCAL_DOFS,), dtype=dtype)
    kernelf0(ffi.from_buffer(b0),
        w_, # This line only works because we specified that b0 is of size LOCAL_DOFS,
        # which happens to also be the size of f._cpp_object
        c_,
        coords_,
        entity_local_index,
        permutation,
        empty_void_pointer(),
    )

    b[:] = G @ b0

In [ ]:
formtype = dolfinx.fem.form_cpp_class(dtype)  # type: ignore
# Gets the number of cells for which each individual core is responsible for.
cells = np.arange(msh.topology.index_map(msh.topology.dim).size_local, dtype=np.int32)

# The 4th argument np.array([...], dtype=np.int8) is the 
# active coefficients array. It lists which indices from the 
# coefficients list should be packed into the w_ pointer that the kernel receives.
integrals = {dolfinx.fem.IntegralType.cell: [
    (0, tabulate_A.address, cells, np.array([0], dtype=np.int8))]}

a_cond = dolfinx.fem.Form( # We are not forming anything yet, this is a recipe
    formtype( # selectes the correct floating-point precision
        spaces=[V_spline._cpp_object, 
                V_spline._cpp_object]
            , # trial and test spaces, determines the size of A_
        integrals=integrals, #this is a dictionary, and we are passing the adress of tabulate_A() here
        coefficients=[C_func._cpp_object
                    ], # weights w_, holds C@T
              constants=[],
              need_permutation_data=False,
              entity_maps=[], 
              mesh=msh._cpp_object)
)

integrals_rhs = {dolfinx.fem.IntegralType.cell: [(0, tabulate_b.address, cells, np.array([0,1], dtype=np.int8))]}
l_cond = dolfinx.fem.Form(
    formtype(
        spaces=[V_spline._cpp_object], # test space, determines the size of b_
        integrals=integrals_rhs, #give the adress of tabulate_b
        coefficients=[f._cpp_object, C_func._cpp_object], # holds the evaluations of f at the correct points, as well as C@T
        constants=[], need_permutation_data=False, entity_maps=[], mesh=msh._cpp_object
    )
)

In [ ]:
from dolfinx.fem.petsc import assemble_matrix, assemble_vector
from petsc4py import PETSc
# a_form = dolfinx.fem.form(a0)
A_cond = assemble_matrix(a_cond, bcs=[])
A_cond.assemble()
A_mat = A_cond
A_mat.setValue(dummy_dof_index, dummy_dof_index, 1., addv=PETSc.InsertMode.INSERT_VALUES)
A_mat.assemble()
A_mat.assemblyBegin()
A_mat.assemblyEnd()

L_cond = assemble_vector(l_cond)
L_cond[dummy_dof_index]=0.
L_cond.assemblyBegin()
L_cond.assemblyEnd()
L_cond.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)


In [ ]:
# Extract the CSR (Compressed Sparse Row) arrays from PETSc
indptr, indices, data = A_mat.getValuesCSR()

# Get the global size of the matrix
shape = A_mat.getSize()

# Create a SciPy CSR matrix
A_scipy = sp.csr_array((data, indices, indptr), shape=shape)

print(f"Matrix shape: {A_scipy.shape}")
print(f"Number of non-zeros: {A_scipy.nnz}")

import matplotlib.pyplot as plt

plt.figure(figsize=(7, 7))
# plt.spy plots the non-zero entries of a matrix
plt.spy(A_scipy, markersize=15, color='blue')
plt.title("Sparsity Pattern of THB-Spline Mass Matrix")
plt.show()

In [ ]:
ksp = PETSc.KSP().create(A_mat.comm)
ksp.setOperators(A_mat)
ksp.setType(PETSc.KSP.Type.CG)
ksp.getPC().setType(PETSc.PC.Type.JACOBI)
u_sol = dolfinx.fem.Function(V_spline)
ksp.solve(L_cond, u_sol.x.petsc_vec)
u_sol.x.scatter_forward()
x_vec=u_sol.x.array
print(f"Solve complete. Reason: {ksp.getConvergedReason()}, Iterations: {ksp.getIterationNumber()}")

In [ ]:
f_sq_form = dolfinx.fem.form(ufl.inner(f, f) * ufl.dx)
f_sq_integral = dolfinx.fem.assemble_scalar(f_sq_form)
f_sq_integral = disconnected_mesh.comm.allreduce(f_sq_integral, op=MPI.SUM)
x_dot_L = L_cond.dot(u_sol.x.petsc_vec)
l2_error_sq = f_sq_integral - x_dot_L
l2_error = np.sqrt(abs(l2_error_sq))

print(f"Squared integral of f: {f_sq_integral}")
print(f"x^T L: {x_dot_L}")
print(f"L2 Error: {l2_error:.5e}")